# Stimulus explorer

Visualize the natural-motion stimulus produced by `motionnet.dataset` and how each
`MovieParams` field changes it. Everything here calls `make_movies` on a few images, so it's
fast and never touches the training cache (except the optional last section).

In [ ]:
from pathlib import Path
import torch

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    REPO = Path("/content/motionnet_cnn")
    if REPO.exists():
        !cd {REPO} && git pull
    else:
        !git clone https://github.com/guardomayas/motionnet.git {REPO}
    !pip install -q -e {REPO}
    import site, importlib
    site.main(); importlib.invalidate_caches()

from motionnet.paths import setup, compose
import seaborn as sns

sns.set_context("talk")
P = setup(IN_COLAB)

In [ ]:
import warnings
from dataclasses import replace, asdict

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from scipy.ndimage import gaussian_filter

from motionnet.dataset import (MovieParams, make_movies, get_data,
                               load_iml, normalize, prefilter, split_images)
from motionnet.dataset.stimulus import _trace, _file_id
from motionnet.dataset.vanhateren import BLUR_TRUNCATE, BOUNDARY_MODE, VH_SHAPE
from motionnet.dataset.plotting import show_hdr

cfg, _ = compose(data="eye31_150fps", model="base", corpus=P.corpus)
STIM = cfg["stimulus"]
BASE = MovieParams(**STIM["movie"])      # warns here if the excursion budget is tight
# ...and every replace() below re-validates; show that warning once, not per variant.
warnings.filterwarnings("ignore", message="excursion budget tight")

train_files, val_files = split_images(STIM["corpus"], **STIM["split"])
FILE = train_files[0]          # the image used for single-sample figures
SEED = STIM["ds_seed_train"]
print(f"{len(train_files)} train / {len(val_files)} val images;  example: {FILE.name}")

## 1. Parameters and derived quantities

`MovieParams` is the single source of truth: the YAML only sets fields; everything below is
derived from them.

In [ ]:
def describe(p):
    H, W = VH_SHAPE
    rows = [
        ("eye", f"{p.eye_size}×{p.eye_size} taps, {p.spacing:.1f} px apart "
                f"({p.deg_per_tap:.2f}°/tap, field {p.eye_size * p.deg_per_tap:.0f}°)"),
        ("scale", f"{p.dpp:.3f} °/px  (Δφ={p.delta_phi_deg}° ≙ {p.blur_px} px)"),
        ("acceptance", f"σ = {p.sigma_px:.2f} px = {p.sigma_px * p.dpp:.2f}°  "
                       f"(Δρ/Δφ = {p.rho_phi_ratio})"),
        ("timing", f"{p.gray_frames} gray + {p.T} motion frames @ {p.fps:g} fps "
                   f"= {p.gray_sec:g} + {p.T / p.fps:.2f} s"),
        ("motion", f"OU, σ_v = {p.vel_std_deg_s} °/s = "
                   f"({p.vel_std_px[0]:.0f}, {p.vel_std_px[1]:.0f}) px/s, "
                   f"t½ = {p.vel_half_life_s} s (τ = {p.tau:.3f} s), ρ = {p.vel_corr}"),
        ("bounds", f"{p.bounds_mode}; excursion limit (x, y) = "
                   f"({p.lim[0]:.0f}, {p.lim[1]:.0f}) px of a {W}×{H} image"),
        ("contrast", f"range {p.contrast_range}, noise σ = {p.noise_std}"),
    ]
    w = max(len(k) for k, _ in rows)
    print("\n".join(f"{k:>{w}}  {v}" for k, v in rows))

describe(BASE)

## 2. Optics: raw image → blurred, log, z-scored → receptor lattice

`prefilter` blurs with the acceptance Gaussian, applies `log1p` + z-score, then converts to
B-spline coefficients. The red square is the eye at the image centre; the dots are its taps
(subsampled so they stay visible). The circle in the zoom is the acceptance FWHM.

In [ ]:
def show_optics(p, file, zoom=None):
    raw = load_iml(file)
    blurred = gaussian_filter(raw, p.sigma_px, mode=BOUNDARY_MODE, truncate=BLUR_TRUNCATE)
    norm = normalize(blurred, p.log_image, p.zscore)
    H, W = VH_SHAPE
    cy, cx = H / 2, W / 2
    off = (np.arange(p.eye_size) - (p.eye_size - 1) / 2) * p.spacing
    zoom = zoom or 1.4 * p.half_px + p.spacing

    fig, axs = plt.subplots(1, 3, figsize=(18, 5), layout="constrained")
    show_hdr(raw, ax=axs[0], title="raw luminance (log display)")
    axs[1].imshow(norm, cmap="gray", vmin=-2.5, vmax=2.5)
    axs[1].set_title(f"blurred σ={p.sigma_px:.1f}px, log + z-score", fontsize=10)
    for ax in axs[:2]:
        ax.add_patch(plt.Rectangle((cx - p.half_px, cy - p.half_px), 2 * p.half_px,
                                   2 * p.half_px, ec="r", fc="none", lw=1.5))
    axs[1].set_axis_off()

    ax = axs[2]
    ax.imshow(norm, cmap="gray", vmin=-2.5, vmax=2.5)
    gx, gy = np.meshgrid(cx + off, cy + off)
    ax.plot(gx, gy, ".", color="tab:orange", ms=3)
    fwhm = p.blur_px * p.rho_phi_ratio
    ax.add_patch(plt.Circle((cx + off[0], cy + off[0]), fwhm / 2, ec="c", fc="none", lw=1.5))
    ax.set_xlim(cx - zoom, cx + zoom); ax.set_ylim(cy + zoom, cy - zoom)
    ax.set_title("receptor lattice (orange) + acceptance FWHM (cyan)", fontsize=10)
    ax.set_axis_off()
    return fig


In [ ]:
_ = show_optics(BASE, FILE)

## 3. One sample, end to end

`render` calls `make_movies` on a single file and also replays the sample's RNG to recover
what `make_movies` doesn't return: the position trace and the gaze-window centre. This
relies on the RNG draw order inside `make_movies` (`_trace`, then the two jitter uniforms).

In [ ]:
def render(p, file=FILE, seed=SEED, progress=False):
    # Movies for one file plus each sample's position trace and gaze centre.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        arrays, stats = make_movies([file], p, seed, progress=progress)
    H, W = VH_SHAPE
    pos, centre, tries = [], [], []
    for rep in range(p.samples_per_image):
        rng = np.random.default_rng((seed, _file_id(file), rep))
        _, ps, n, _ = _trace(p, rng)
        centre.append((W / 2 + rng.uniform(-p.start_jitter, p.start_jitter),
                       H / 2 + rng.uniform(-p.start_jitter, p.start_jitter)))
        pos.append(ps); tries.append(n)
    arrays = {k: np.asarray(v, np.float32 if k == "movie" else v.dtype)
              for k, v in arrays.items()}
    return dict(**arrays, pos_px=np.stack(pos), centre=np.array(centre),
                tries=np.array(tries), stats=stats, p=p, file=file)


def snr_label(rms, noise_std):
    # Amplitude SNR of clean signal (contrast_rms) to the added pixel noise.
    if noise_std <= 0:
        return "SNR ∞"
    snr = rms / noise_std
    return f"SNR {snr:.2g} ({20 * np.log10(snr):.0f} dB)"


def tap_extent(p):
    h = (p.eye_size - 1) / 2 * p.deg_per_tap + p.deg_per_tap / 2
    return [-h, h, -h, h]


def show_sample(r, i=0, frames=None, title=None):
    p, G = r["p"], r["p"].gray_frames
    movie = r["movie"][i, G:, 0]
    vel = r["vel_deg_s"][i, G:]
    pos = r["pos_px"][i] * p.dpp
    t_s = np.arange(p.T) / p.fps
    frames = frames if frames is not None else np.linspace(0, p.T - 1, 4).astype(int)
    vmin, vmax = np.percentile(movie, [1, 99])

    fig = plt.figure(figsize=(4 * len(frames), 8), layout="constrained")
    gs = fig.add_gridspec(2, len(frames))
    for k, f in enumerate(frames):
        ax = fig.add_subplot(gs[0, k])
        ax.imshow(movie[f], cmap="gray", vmin=vmin, vmax=vmax,
                  interpolation="nearest", extent=tap_extent(p))
        ax.set_title(f"t = {f / p.fps:.2f} s", fontsize=12)
        if k: ax.set_yticks([])
    sub = gs[1, :].subgridspec(1, 3)
    ax = fig.add_subplot(sub[0])
    ax.plot(t_s, pos[:, 0], label="x"); ax.plot(t_s, pos[:, 1], label="y")
    for f in frames: ax.axvline(f / p.fps, color="0.7", lw=0.8)
    ax.set(xlabel="time (s)", ylabel="position (deg)"); ax.legend(fontsize=10)
    ax = fig.add_subplot(sub[1])
    ax.plot(pos[:, 0], pos[:, 1], lw=1)
    ax.scatter(*pos[0], c="g", s=30, zorder=3, label="start")
    ax.scatter(*pos[-1], c="r", s=30, zorder=3, label="end")
    ax.set(xlabel="x (deg)", ylabel="y (deg)", aspect="equal"); ax.legend(fontsize=10)
    ax = fig.add_subplot(sub[2])
    ax.plot(t_s, vel[:, 0], label="vx"); ax.plot(t_s, vel[:, 1], label="vy")
    ax.axhline(0, ls="--", color="k", lw=0.5)
    ax.set(xlabel="time (s)", ylabel="velocity (deg/s)"); ax.legend(fontsize=10)
    fig.suptitle(title or f"{r['file'].name}  sample {i}  "
                 f"(gain {r['contrast_gain'][i]:.2f}, rms {r['contrast_rms'][i]:.2f}, "
                 f"{snr_label(r['contrast_rms'][i], p.noise_std)}, draws {r['tries'][i]})", fontsize=13)
    return fig

R = render(BASE)
_ = show_sample(R, 0)

### Animation

Left: the source (prefiltered) image with the gaze window following the trace. Middle: what the
eye sees, including the gray lead-in. Right: velocity with a time cursor.

In [ ]:
plt.rcParams["animation.embed_limit"] = 50   # MB; jshtml embeds every frame as a PNG


def animate(r, i=0, stride=3, dpi=60):
    p, G = r["p"], r["p"].gray_frames
    movie = r["movie"][i, :, 0]
    vel = r["vel_deg_s"][i]
    pos = np.concatenate([np.zeros((G, 2), np.float32), r["pos_px"][i]])
    cx, cy = r["centre"][i]
    img = normalize(gaussian_filter(load_iml(r["file"]), p.sigma_px, mode=BOUNDARY_MODE,
                                    truncate=BLUR_TRUNCATE), p.log_image, p.zscore)
    t_s = (np.arange(G + p.T) - G) / p.fps
    vmin, vmax = np.percentile(movie[G:], [1, 99])
    hp = p.half_px

    fig, axs = plt.subplots(1, 3, figsize=(16, 4.5), dpi=dpi, layout="constrained")
    axs[0].imshow(img, cmap="gray", vmin=-2.5, vmax=2.5)
    axs[0].plot(cx + pos[G:, 0], cy + pos[G:, 1], "y", lw=0.8)
    rect = plt.Rectangle((cx - hp, cy - hp), 2 * hp, 2 * hp, ec="r", fc="none", lw=1.5)
    axs[0].add_patch(rect); axs[0].set_axis_off()
    axs[0].set_title("source + gaze window", fontsize=12)
    im = axs[1].imshow(movie[0], cmap="gray", vmin=vmin, vmax=vmax,
                       interpolation="nearest", extent=tap_extent(p))
    axs[1].set(xlabel="azimuth (deg)", ylabel="elevation (deg)")
    ttl = axs[1].set_title("", fontsize=12)
    axs[2].plot(t_s, vel[:, 0], label="vx", lw=1); axs[2].plot(t_s, vel[:, 1], label="vy", lw=1)
    axs[2].axhline(0, color="k", lw=0.5); axs[2].axvspan(t_s[0], 0, color="0.9")
    cursor = axs[2].axvline(t_s[0], color="r", lw=1.5)
    axs[2].set(xlabel="time (s)", ylabel="velocity (deg/s)"); axs[2].legend(fontsize=10)

    def update(t):
        im.set_data(movie[t])
        rect.set_xy((cx + pos[t, 0] - hp, cy + pos[t, 1] - hp))
        cursor.set_xdata([t_s[t]] * 2)
        ttl.set_text(f"t = {t_s[t]:+.2f} s" + ("  (gray)" if t < G else ""))
        return im, rect, cursor, ttl

    anim = FuncAnimation(fig, update, frames=range(0, len(movie), stride),
                         interval=1000 * stride / p.fps)
    plt.close(fig)
    return anim

display(HTML(animate(R, 0).to_jshtml(default_mode="loop")))

## 4. Parameter sweeps

Each sweep changes **one** field with `dataclasses.replace` and re-renders the same file with the
same seed, so differences are due to that field alone. `sweep_frames` shows the first motion
frame per value; `sweep_traces` overlays the velocity traces.

In [ ]:
def variants(field, values, base=BASE, **fixed):
    # One MovieParams per value; `fixed` overrides apply to all (e.g. noise off).
    return [replace(base, samples_per_image=1, **fixed, **{field: v}) for v in values]


def sweep_frames(field, values, frame=0, fmt="{}", **fixed):
    ps = variants(field, values, **fixed)
    fig, axs = plt.subplots(1, len(ps), figsize=(3.6 * len(ps), 4), layout="constrained")
    for ax, p, v in zip(axs, ps, values):
        r = render(p)
        m = r["movie"][0, p.gray_frames + frame, 0]
        lo, hi = np.percentile(r["movie"][0, p.gray_frames:], [1, 99])
        ax.imshow(m, cmap="gray", vmin=lo, vmax=hi, interpolation="nearest",
                  extent=tap_extent(p))
        rms = r["contrast_rms"][0]
        ax.set_title(f"{field} = {fmt.format(v)}\nrms {rms:.2f}, "
                     f"{snr_label(rms, p.noise_std)}", fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
    return fig


def sweep_traces(field, values, fmt="{}", n=3, **fixed):
    ps = variants(field, values, **fixed)
    fig, axs = plt.subplots(2, len(ps), figsize=(4 * len(ps), 6), sharex=True,
                            sharey="row", layout="constrained", squeeze=False)
    for k, (p, v) in enumerate(zip(ps, values)):
        r = render(replace(p, samples_per_image=n))
        t_s = np.arange(p.T) / p.fps
        for i in range(n):
            axs[0, k].plot(t_s, r["vel_deg_s"][i, p.gray_frames:, 0], lw=1)
            axs[1, k].plot(t_s, r["pos_px"][i, :, 0] * p.dpp, lw=1)
        axs[0, k].set_title(f"{field} = {fmt.format(v)}", fontsize=11)
        axs[1, k].set_xlabel("time (s)")
    axs[0, 0].set_ylabel("vx (deg/s)"); axs[1, 0].set_ylabel("x (deg)")
    return fig

### Optics: acceptance blur and lattice

In [ ]:
# blur_px sets both Δρ and the deg/px scale (Δφ ≙ blur_px px); spacing follows it.
sweep_frames("blur_px", [6, 9, 13, 18], noise_std=0.0);

In [ ]:
# Lattice spacing alone: under-sampling (spacing > blur_px) vs over-sampling.
sweep_frames("spacing_px", [5, 9, 13, 20], noise_std=0.0);

In [ ]:
sweep_frames("rho_phi_ratio", [0.5, 1.08, 1.5, 2.5], noise_std=0.0);

In [ ]:
sweep_frames("eye_size", [11, 21, 31, 41], noise_std=0.0);

In [ ]:
# Photoreceptor nonlinearity / normalization
fig, axs = plt.subplots(1, 4, figsize=(11, 4), layout="constrained")
for ax, kw in zip(axs, [dict(log_image=True, zscore=True),
                        dict(log_image=False, zscore=True),
                        dict(log_image=True, zscore=False),
                        dict(log_image=False, zscore=False)]):
    p = replace(BASE, samples_per_image=1, noise_std=0.0, **kw)
    m = render(p)["movie"][0, p.gray_frames, 0]
    ax.imshow(m, cmap="gray", interpolation="nearest")
    ax.set_title(", ".join(f"{k}={v}" for k, v in kw.items()), fontsize=10)
    ax.set_axis_off()

### Contrast and noise

In [ ]:
sweep_frames("noise_std", [0.0, 0.1, 0.3, 1.0], fmt="{:g}");

In [ ]:
# contrast_range draws a per-sample gain; equal endpoints make it deterministic here.
sweep_frames("contrast_range", [(0.1, 0.1), (0.3, 0.3), (1.0, 1.0), (3.0, 3.0)],
             fmt="{0[0]:g}", noise_std=0.1);

### Motion statistics

In [ ]:
sweep_traces("vel_std_deg_s", [30, 70, 120, 200], n=4)

In [ ]:
sweep_traces("vel_half_life_s", [0.02, 0.1, 0.2, 0.5], n=4)

In [ ]:
# vel_corr couples x and y velocity: the trajectory tilts along the diagonal.
ps = variants("vel_corr", [-0.9, 0.0, 0.9], vel_std_deg_s=(100, 100))
fig, axs = plt.subplots(1, len(ps), figsize=(13, 4.5), layout="constrained")
for ax, p in zip(axs, ps):
    r = render(replace(p, samples_per_image=12))
    for xy in r["pos_px"] * p.dpp:
        ax.plot(*xy.T, lw=0.8)
    ax.set(title=f"vel_corr = {p.vel_corr}", xlabel="x (deg)", aspect="equal")
axs[0].set_ylabel("y (deg)");

### Bounds handling

With a large velocity spread the trace often leaves the image. `reject` redraws (so kept
velocities are conditioned on staying in bounds); `reflect` folds the position at the walls,
which shows up as velocity sign flips. Dashed lines are the excursion limits.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 8), layout="constrained")
for col, mode in enumerate(["reject", "reflect"]):
    p = replace(BASE, vel_std_deg_s=(300, 300), bounds_mode=mode, samples_per_image=6)
    r = render(p)
    t_s = np.arange(p.T) / p.fps
    for i in range(p.samples_per_image):
        axs[0, col].plot(t_s, r["pos_px"][i, :, 0], lw=1)
        axs[1, col].plot(t_s, r["vel_deg_s"][i, p.gray_frames:, 0], lw=1)
    for s in (-1, 1):
        axs[0, col].axhline(s * p.lim[0], ls="--", color="k", lw=0.8)
    axs[0, col].set_title(f"{mode}: acceptance {r['stats']['acceptance']:.0%}, "
                          f"fallbacks {r['stats']['fallbacks']}", fontsize=12)
    axs[1, col].set_xlabel("time (s)")
axs[0, 0].set_ylabel("x (source px)"); axs[1, 0].set_ylabel("vx (deg/s)");

## 5. Ensemble statistics

Render many samples from a handful of training images with the base config and check the
velocity marginals and autocorrelation against the OU targets (σ_v and exp(−t/τ)).
Rejection sampling narrows the marginal when the excursion budget is tight.

In [ ]:
N_FILES = 12
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    ens, ens_stats = make_movies(train_files[:N_FILES], BASE, SEED)
G = BASE.gray_frames
V = ens["vel_deg_s"][:, G:]                       # (N, T, 2)
print(f"{len(V)} samples, acceptance {ens_stats['acceptance']:.0%}, "
      f"fallbacks {ens_stats['fallbacks']}")

def autocorr(x, max_lag):
    x = x - x.mean(1, keepdims=True)
    c = np.array([(x[:, :x.shape[1] - k] * x[:, k:]).mean() for k in range(max_lag)])
    return c / c[0]

fig, axs = plt.subplots(1, 3, figsize=(17, 4.5), layout="constrained")
for d, name in enumerate("xy"):
    s = BASE.vel_std_deg_s[d]
    v = V[..., d].ravel()
    axs[d].hist(v, bins=60, density=True, alpha=0.6, label=f"empirical σ={v.std():.0f}")
    g = np.linspace(-4 * s, 4 * s, 200)
    axs[d].plot(g, np.exp(-g**2 / (2 * s**2)) / (s * np.sqrt(2 * np.pi)), "k",
                label=f"target σ={s:g}")
    axs[d].set(xlabel=f"v{name} (deg/s)", title=f"v{name} marginal"); axs[d].legend(fontsize=10)

lags = np.arange(min(BASE.T, int(4 * BASE.tau * BASE.fps)))
for d, name in enumerate("xy"):
    axs[2].plot(lags / BASE.fps, autocorr(V[..., d], len(lags)), label=f"v{name}")
axs[2].plot(lags / BASE.fps, np.exp(-lags / BASE.fps / BASE.tau), "k--", label="exp(-t/τ)")
axs[2].axvline(BASE.vel_half_life_s, color="0.6", lw=0.8)
axs[2].set(xlabel="lag (s)", ylabel="autocorrelation", title="velocity ACF")
axs[2].legend(fontsize=10);

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
axs[0].hist(ens["contrast_rms"], bins=20)
axs[0].set(xlabel="contrast rms (pre-noise)", ylabel="samples")
if BASE.noise_std > 0:
    # SNR = rms / noise_std is a rescaling of the same axis.
    k = BASE.noise_std
    axs[0].secondary_xaxis("top", functions=(lambda x: x / k, lambda s: s * k)) \
          .set_xlabel(f"SNR (noise σ = {k:g})")
else:
    axs[0].set_title("noise_std = 0: SNR ∞", fontsize=12)
axs[1].hist2d(V[..., 0].ravel(), V[..., 1].ravel(), bins=60, cmap="magma")
axs[1].set(xlabel="vx (deg/s)", ylabel="vy (deg/s)", aspect="equal", title="joint velocity");

## 6. (Optional) the cached training set

`get_data` builds or loads the full train/val tensors for this config (slow the first time).

In [ ]:
LOAD_CACHE = True
if LOAD_CACHE:
    data = get_data(STIM, P.movies, "cpu")
    print(data.info)
    v = data.train["vel_deg_s"][:, data.info.gray_frames:]
    print("train vel std (x, y):", v.reshape(-1, 2).std(0).numpy().round(1))